# Results Summary — Q-SITE 2026 Open Challenges

**Team:** WestQuantOpen

---

## Overview

This notebook summarizes the key results from both challenge tracks.

| Track | Metric | Before | After | Improvement |
|-------|--------|--------|-------|-------------|
| Computational | Total score | 119.0 | 86.5 | 27.3% |
| Scientific | Classifier accuracy | 25.85% | 70.1% | +44.3% |
| Scientific | Antiphase VQE ΔE | 2.162 | 0.021 | 99.0% |
| Scientific | Interior point accuracy | — | 100% | — |

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import json

# === Computational Results ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score by benchmark
benchmarks = ['ghz_star', 'chain_trotter', 'ladder_trotter', 'qaoa_random', 'dense_random', 'vqe_layers']
baseline = [11.0, 4.5, 10.5, 20.0, 70.0, 3.0]
with open('../results/best.json') as f:
    best = json.load(f)
final = [best['per_benchmark'][b]['score'] for b in benchmarks]

x = np.arange(len(benchmarks))
width = 0.35
axes[0].bar(x - width/2, baseline, width, label='Baseline', color='#e74c3c', alpha=0.8)
axes[0].bar(x + width/2, final, width, label='Final', color='#2ecc71', alpha=0.8)
axes[0].set_ylabel('Score (lower is better)')
axes[0].set_title('Computational: Benchmark Scores', fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(benchmarks, rotation=45, ha='right')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Score progression
stages = ['Baseline', 'V1', 'V2', '12h', 'Final']
totals = [119.0, 99.5, 97.0, 89.0, 86.5]
axes[1].plot(stages, totals, 'o-', linewidth=2, markersize=10, color='#3498db')
axes[1].fill_between(range(len(stages)), totals, alpha=0.2, color='#3498db')
axes[1].set_ylabel('Total Score')
axes[1].set_title('Computational: Score Progression', fontsize=13, fontweight='bold')
axes[1].grid(alpha=0.3)
for i, t in enumerate(totals):
    axes[1].annotate(f'{t:.1f}', (i, t), textcoords="offset points", xytext=(0, 10), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../presentations/images/computational_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Scientific Results ===
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Classifier accuracy
phases = ['Ferro', 'Anti', 'Paramag', 'Overall', 'Interior']
before = [68.5, 0.0, 0.0, 25.85, 0.0]
after = [92.3, 93.3, 81.3, 70.1, 100.0]
x = np.arange(len(phases))
width = 0.35
axes[0].bar(x - width/2, before, width, label='Before', color='#e74c3c', alpha=0.8)
axes[0].bar(x + width/2, after, width, label='After', color='#2ecc71', alpha=0.8)
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Classifier Accuracy', fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(phases, rotation=45, ha='right')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim(0, 110)

# VQE comparison
points = ['Ferro', 'Antiphase', 'Paramag']
old_vqe = [0.002, 2.162, 1.858]
new_vqe = [0.003, 0.021, 0.042]
x = np.arange(len(points))
axes[1].bar(x - width/2, old_vqe, width, label='Standard', color='#e74c3c', alpha=0.8)
axes[1].bar(x + width/2, new_vqe, width, label='Phase-Adapted', color='#2ecc71', alpha=0.8)
axes[1].set_ylabel('ΔE (log scale)')
axes[1].set_title('VQE Energy Accuracy', fontsize=13, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(points)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)
axes[1].set_yscale('log')

# Noise degradation
noise_levels = [0.0, 0.01, 0.05]
ferro_s0 = [7.896, 7.362, 5.558]
anti_s = [3.901, 3.677, 2.918]
para_x = [0.933, 0.896, 0.758]
axes[2].plot(noise_levels, ferro_s0, 'o-', linewidth=2, label='Ferro S(0)', color='#e74c3c')
axes[2].plot(noise_levels, anti_s, 's-', linewidth=2, label='Anti S(π/2)', color='#3498db')
axes[2].plot(noise_levels, para_x, '^-', linewidth=2, label='Para ⟨X⟩', color='#2ecc71')
axes[2].set_xlabel('Noise p')
axes[2].set_ylabel('Order Parameter')
axes[2].set_title('Noise Degradation', fontsize=13, fontweight='bold')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../presentations/images/scientific_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Innovations

### Computational
1. **Edge-meeting router** — Novel dual-ended routing that initiates paths from both gate endpoints
2. **LNS for circuit compilation** — First method to break the SA+SABRE plateau on dense_random
3. **A* certificates** — Exhaustive search confirms GHZ (7.0) and Ladder (6.5) are near-optimal

### Scientific
1. **Structure factor correction** — Fixed systematic -1.0 offset from missing diagonal terms
2. **Antiphase wavevector correction** — q* = π/2 (period 4: ↑↑↓↓), not π
3. **Phase-adapted reference-state HVA** — U_HVA(θ)|ψ_ref⟩ solves antiphase VQE (ΔE 2.16 → 0.02)
4. **Dual-signal phase detection** — Observable classifier + variational basin signal provide independent phase identification

## Research Question

The project evolved from "map the ANNNI phase diagram" to:

> **"How do representation choice and gate noise jointly affect quantum phase detection?"**

This reveals that the interplay between ansatz initialization, physical observables, and noise creates a multi-dimensional phase detection problem — more interesting than simply showing three color maps.